# Кастомная CNN для классификации real vs AI face

Ноутбук с полностью кастомной CNN, без готовых архитектур.


# Библиотеки и конфиг
Здесь подключаются все основные библиотеки для работы с данными, изображениями и PyTorch. Также создаётся класс `CFG`, в котором собраны гиперпараметры обучения, благодаря чему модель проще настраивать и запускать повторно.

In [1]:
import os
import gc
import cv2
import copy
import random
from pathlib import Path
from contextlib import nullcontext

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import albumentations as A
from albumentations.pytorch import ToTensorV2

cv2.setNumThreads(0)


class CFG:
    seed = 42
    n_folds = 5
    epochs = 30
    patience = 5

    img_size = 256
    batch_size = 32
    num_workers = 2

    lr = 1e-3
    weight_decay = 1e-4
    max_grad_norm = 5.0
    label_smoothing = 0.01
    ema_decay = 0.999

    use_amp = True
    use_tta = True

    train_csv = "/kaggle/input/datasets/tomatka/aifaceclassification/train_solution.csv"
    train_dir = "/kaggle/input/datasets/tomatka/aifaceclassification/train_images"
    test_dir = "/kaggle/input/datasets/tomatka/aifaceclassification/test_images"
    out_dir = "/kaggle/working"


os.makedirs(CFG.out_dir, exist_ok=True)


# Проверка доступности GPU
Функция `get_runtime_device()` проверяет, доступна ли GPU, пытается выполнить небольшой вычислительный тест на CUDA и, если всё успешно, выбирает `cuda`. Если GPU недоступна или возникает ошибка совместимости, используется CPU.

In [2]:
def get_runtime_device():
    if not torch.cuda.is_available():
        print("CUDA не найдена, обучение пойдет на CPU.")
        return torch.device("cpu")

    try:
        dev = torch.device("cuda")
        gpu_name = torch.cuda.get_device_name(0)
        print(f"Найдена GPU: {gpu_name}")

        x = torch.randn(2, 3, 32, 32, device=dev)
        conv = nn.Conv2d(3, 8, kernel_size=3, padding=1).to(dev)
        with torch.no_grad():
            y = conv(x)
            _ = y.mean().item()

        print("CUDA smoke-test пройден.")
        return dev

    except Exception as e:
        print("\n[WARNING] CUDA недоступна для вычислений в текущем окружении.")
        return torch.device("cpu")


device = get_runtime_device()
USE_AMP = bool(CFG.use_amp and device.type == "cuda")
PIN_MEMORY = bool(device.type == "cuda")

print("Устройство:", device)
print("AMP:", USE_AMP)


Найдена GPU: Tesla T4
CUDA smoke-test пройден.
Устройство: cuda
AMP: True


# Фиксация seed и подготовка таблиц с данными
В этом блоке:
- фиксируются сиды через `seed_everything()` для более стабильных результатов;
- автоматически определяется, какие столбцы в CSV отвечают за `id` и `target`;
- по `id` собираются реальные пути к изображениям;
- формируются `train_df` и `test_df` с удобной структурой: `id`, `path`, `label`.

Приводим данные к формату, удобному для последующей загрузки в Dataset и DataLoader.

In [3]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = (device.type == "cuda")


def infer_columns(df: pd.DataFrame):
    cols = list(df.columns)
    target_candidates = [c for c in cols if c.lower() in {"target", "label", "class", "y"}]
    target_col = target_candidates[0] if target_candidates else cols[-1]
    id_col = [c for c in cols if c != target_col][0]
    return id_col, target_col


def resolve_img_path(base_dir: str, file_id: str):
    p = Path(base_dir) / str(file_id)
    if p.exists():
        return str(p)

    stem = str(file_id)
    for ext in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
        p = Path(base_dir) / f"{stem}{ext}"
        if p.exists():
            return str(p)

    raise FileNotFoundError(f"Не найден файл для id={file_id} в {base_dir}")


def build_dataframes(train_csv, train_dir, test_dir):
    df = pd.read_csv(train_csv)
    id_col, target_col = infer_columns(df)

    df[id_col] = df[id_col].astype(str)
    train_df = pd.DataFrame({
        "id": df[id_col].astype(str),
        "path": df[id_col].astype(str).apply(lambda x: resolve_img_path(train_dir, x)),
        "label": df[target_col].astype(int)
    })

    train_has_ext = train_df["id"].map(lambda x: Path(x).suffix != "").mean() > 0.8

    test_files = sorted([
        p for p in Path(test_dir).iterdir()
        if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    ])

    test_ids = [p.name if train_has_ext else p.stem for p in test_files]
    test_df = pd.DataFrame({
        "id": test_ids,
        "path": [str(p) for p in test_files]
    })

    return train_df, test_df, id_col, target_col


# Аугментации изображений и класс датасета
Здесь задаются функции аугментации для train и valid:
- случайный crop;
- горизонтальный flip;
- небольшие геометрические преобразования;
- шум, blur, компрессия, `ColorJitter`;
- нормализация и перевод изображения в тензор.

Также объявляется класс `FaceDataset`, который читает изображение с диска, переводит его в RGB, применяет трансформации и возвращает либо `(image, label)`, либо только `image` для теста. Стоит отметить, что с аугментацией нужно работать осторожней, так как для этой задачи у сгенерированных лиц часто заметны артефакты именно в текстуре кожи, волосах, границах, мелких деталях и так далее. Чтобы не потерять эти признаки, не стоит аугментировать слишком агрессивно.

In [4]:
def _random_resized_crop(img_size):
    try:
        return A.RandomResizedCrop(
            size=(img_size, img_size),
            scale=(0.88, 1.0),
            ratio=(0.95, 1.05),
            p=1.0,
        )
    except Exception:
        return A.RandomResizedCrop(
            height=img_size,
            width=img_size,
            scale=(0.88, 1.0),
            ratio=(0.95, 1.05),
            p=1.0,
        )


def _gauss_noise():
    try:
        return A.GaussNoise(
            std_range=(0.02, 0.08),
            mean_range=(0.0, 0.0),
            p=1.0,
        )
    except Exception:
        try:
            return A.GaussNoise(var_limit=(10.0, 40.0), p=1.0)
        except Exception:
            return A.GaussNoise(p=1.0)


def _image_compression():
    try:
        return A.ImageCompression(quality_range=(70, 100), p=1.0)
    except Exception:
        return A.ImageCompression(quality_lower=70, quality_upper=100, p=1.0)


def get_train_transforms(img_size):
    return A.Compose([
        _random_resized_crop(img_size),
        A.HorizontalFlip(p=0.5),
        A.Affine(
            scale=(0.94, 1.06),
            translate_percent={"x": (-0.04, 0.04), "y": (-0.04, 0.04)},
            rotate=(-10, 10),
            shear=(-3, 3),
            border_mode=cv2.BORDER_REFLECT_101,
            p=0.45,
        ),
        A.OneOf([
            _gauss_noise(),
            A.ISONoise(color_shift=(0.01, 0.03), intensity=(0.1, 0.3), p=1.0),
        ], p=0.20),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 5), p=1.0),
            A.MotionBlur(blur_limit=3, p=1.0),
            A.MedianBlur(blur_limit=3, p=1.0),
        ], p=0.10),
        A.OneOf([
            _image_compression(),
            A.ColorJitter(brightness=0.08, contrast=0.08, saturation=0.08, hue=0.04, p=1.0),
        ], p=0.20),
        A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ToTensorV2(),
    ])


def get_valid_transforms(img_size):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ToTensorV2(),
    ])


class FaceDataset(Dataset):
    def __init__(self, df, transforms=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        path = self.df.loc[idx, "path"]
        image = cv2.imread(path)
        if image is None:
            raise FileNotFoundError(f"cv2 не смог прочитать файл: {path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transforms is not None:
            image = self.transforms(image=image)["image"]

        if self.is_test:
            return image

        label = torch.tensor(self.df.loc[idx, "label"], dtype=torch.float32)
        return image, label


# Архитектура CNN и EMA
В этом блоке описываются все основные строительные элементы модели:
- `SEBlock` для channel attention;
- `ConvBNAct` как базовый сверточный блок;
- `ResidualSEBlock` с residual-соединением и attention;
- `CustomFaceCNN` — основная сеть для бинарной классификации.

Особенность модели в том, что она работает не только с RGB-изображением, но и строит high-pass представление через лапласиан. Это помогает выделять высокочастотные артефакты, которые часто встречаются у AI-сгенерированных лиц.

Также реализован класс `EMA`, который поддерживает экспоненциальное усреднение весов модели.

In [5]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(16, channels // reduction)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Conv2d(channels, hidden, kernel_size=1)
        self.fc2 = nn.Conv2d(hidden, channels, kernel_size=1)

    def forward(self, x):
        w = self.pool(x)
        w = F.silu(self.fc1(w), inplace=True)
        w = torch.sigmoid(self.fc2(w))
        return x * w


class ConvBNAct(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=None):
        super().__init__()
        if p is None:
            p = k // 2
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class ResidualSEBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = ConvBNAct(in_ch, out_ch, k=3, s=stride)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.se = SEBlock(out_ch, reduction=8)
        self.dropout = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()

        self.skip = nn.Identity()
        if stride != 1 or in_ch != out_ch:
            self.skip = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        identity = self.skip(x)
        out = self.conv1(x)
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        out = self.dropout(out)
        out = out + identity
        out = self.act(out)
        return out


class CustomFaceCNN(nn.Module):
    def __init__(self):
        super().__init__()

        lap = torch.tensor([[0, -1, 0],
                            [-1, 4, -1],
                            [0, -1, 0]], dtype=torch.float32).view(1, 1, 3, 3)
        self.register_buffer("lap_kernel", lap)

        self.rgb_stem = nn.Sequential(
            ConvBNAct(3, 32, k=3, s=2),
            ConvBNAct(32, 32, k=3, s=1),
        )

        self.hp_stem = nn.Sequential(
            ConvBNAct(1, 16, k=3, s=2),
            ConvBNAct(16, 16, k=3, s=1),
        )

        self.fuse = ConvBNAct(48, 48, k=1, s=1, p=0)

        self.stage1 = nn.Sequential(
            ResidualSEBlock(48, 64, stride=1, dropout=0.00),
            ResidualSEBlock(64, 64, stride=1, dropout=0.00),
        )
        self.stage2 = nn.Sequential(
            ResidualSEBlock(64, 96, stride=2, dropout=0.00),
            ResidualSEBlock(96, 96, stride=1, dropout=0.00),
        )
        self.stage3 = nn.Sequential(
            ResidualSEBlock(96, 144, stride=2, dropout=0.05),
            ResidualSEBlock(144, 144, stride=1, dropout=0.05),
        )
        self.stage4 = nn.Sequential(
            ResidualSEBlock(144, 224, stride=2, dropout=0.10),
            ResidualSEBlock(224, 224, stride=1, dropout=0.10),
        )

        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.head = nn.Sequential(
            nn.Linear(224 * 2, 256),
            nn.SiLU(inplace=True),
            nn.Dropout(0.35),
            nn.Linear(256, 1)
        )

    def make_highpass(self, x):
        gray = 0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3]
        hp = F.conv2d(gray, self.lap_kernel, padding=1)
        hp = torch.clamp(hp, -4.0, 4.0) / 4.0
        return hp

    def forward(self, x):
        hp = self.make_highpass(x)

        rgb = self.rgb_stem(x)
        hp = self.hp_stem(hp)

        x = torch.cat([rgb, hp], dim=1)
        x = self.fuse(x)

        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)

        x_avg = self.avg_pool(x).flatten(1)
        x_max = self.max_pool(x).flatten(1)
        x = torch.cat([x_avg, x_max], dim=1)
        x = self.head(x)
        return x


class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        for name, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[name] = p.data.clone()

    @torch.no_grad()
    def update(self, model):
        for name, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[name].mul_(self.decay).add_(p.data, alpha=1.0 - self.decay)

    def apply_shadow(self, model):
        self.backup = {}
        for name, p in model.named_parameters():
            if p.requires_grad:
                self.backup[name] = p.data.clone()
                p.data.copy_(self.shadow[name])

    def restore(self, model):
        for name, p in model.named_parameters():
            if p.requires_grad:
                p.data.copy_(self.backup[name])
        self.backup = {}


# Функции обучения, валидации и предсказания
В этом блоке реализованы:
- `smooth_labels()` — label smoothing для более мягких целевых значений;
- `find_best_threshold()` — подбор лучшего порога по F1-score;
- `get_autocast_context()` — включение mixed precision при наличии CUDA;
- `train_one_epoch()` — обучение модели одну эпоху;
- `validate()` — валидация модели на отложенной выборке;
- `predict_proba()` — получение вероятностей для valid/test, включая TTA через horizontal flip.

In [6]:
def smooth_labels(targets, smoothing=0.01):
    return targets * (1.0 - smoothing) + 0.5 * smoothing


def find_best_threshold(y_true, y_prob):
    thresholds = np.linspace(0.05, 0.95, 181)
    best_t, best_f1 = 0.5, -1.0
    for t in thresholds:
        preds = (y_prob >= t).astype(np.int32)
        score = f1_score(y_true, preds)
        if score > best_f1:
            best_f1 = score
            best_t = t
    return best_t, best_f1


def get_autocast_context():
    if USE_AMP:
        return torch.amp.autocast(device_type="cuda", enabled=True)
    return nullcontext()


def train_one_epoch(model, loader, optimizer, scheduler, criterion, scaler, ema):
    model.train()
    running_loss = 0.0

    pbar = tqdm(loader, total=len(loader), leave=False)
    for images, targets in pbar:
        images = images.to(device, non_blocking=PIN_MEMORY)
        targets = targets.view(-1, 1).to(device, non_blocking=PIN_MEMORY)
        targets_smooth = smooth_labels(targets, CFG.label_smoothing)

        optimizer.zero_grad(set_to_none=True)

        with get_autocast_context():
            logits = model(images)
            loss = criterion(logits, targets_smooth)

        if USE_AMP:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)

            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            new_scale = scaler.get_scale()

            optimizer_step_happened = new_scale >= old_scale
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
            optimizer.step()
            optimizer_step_happened = True

        if scheduler is not None and optimizer_step_happened:
            scheduler.step()

        ema.update(model)

        running_loss += loss.item() * images.size(0)
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return running_loss / len(loader.dataset)


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    all_probs, all_targets = [], []

    for images, targets in tqdm(loader, total=len(loader), leave=False):
        images = images.to(device, non_blocking=PIN_MEMORY)
        targets = targets.view(-1, 1).to(device, non_blocking=PIN_MEMORY)

        with get_autocast_context():
            logits = model(images)
            loss = criterion(logits, targets)

        probs = torch.sigmoid(logits)

        running_loss += loss.item() * images.size(0)
        all_probs.append(probs.squeeze(1).cpu().numpy())
        all_targets.append(targets.squeeze(1).cpu().numpy())

    all_probs = np.concatenate(all_probs)
    all_targets = np.concatenate(all_targets).astype(np.int32)
    return running_loss / len(loader.dataset), all_probs, all_targets


@torch.no_grad()
def predict_proba(model, loader, tta=True):
    model.eval()
    all_probs = []

    for batch in tqdm(loader, total=len(loader), leave=False):
        if isinstance(batch, (list, tuple)):
            images = batch[0]
        else:
            images = batch

        images = images.to(device, non_blocking=PIN_MEMORY)

        with get_autocast_context():
            logits = model(images)
            probs = torch.sigmoid(logits)

            if tta:
                logits_flip = model(torch.flip(images, dims=[3]))
                probs_flip = torch.sigmoid(logits_flip)
                probs = 0.5 * (probs + probs_flip)

        all_probs.append(probs.squeeze(1).cpu().numpy())

    return np.concatenate(all_probs)


# Основной пайплайн обучения с кросс-валидацией
Финальный блок, где полностью собирается решение. Связываем все подготовленные ранее части в единый пайплайн и на выходе получаем финальный `submission.csv` для соревнования.

In [7]:
def main():
    seed_everything(CFG.seed)

    print("\nИспользуемое устройство:", device)
    print("AMP включен:", USE_AMP)
    print("Пути:")
    print("train_csv =", CFG.train_csv)
    print("train_dir =", CFG.train_dir)
    print("test_dir  =", CFG.test_dir)
    print("out_dir   =", CFG.out_dir)

    train_df, test_df, id_col, target_col = build_dataframes(
        CFG.train_csv, CFG.train_dir, CFG.test_dir
    )

    print(f"\nTrain size: {len(train_df)}")
    print(f"Test  size: {len(test_df)}")
    print(train_df["label"].value_counts(dropna=False))

    test_ds = FaceDataset(test_df, transforms=get_valid_transforms(CFG.img_size), is_test=True)
    test_loader = DataLoader(
        test_ds,
        batch_size=CFG.batch_size,
        shuffle=False,
        num_workers=CFG.num_workers,
        pin_memory=PIN_MEMORY,
        drop_last=False,
        persistent_workers=CFG.num_workers > 0,
    )

    oof_probs = np.zeros(len(train_df), dtype=np.float32)
    test_probs_folds = []
    fold_thresholds = []

    skf = StratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, train_df["label"].values), 1):
        print(f"\n{'=' * 20} FOLD {fold}/{CFG.n_folds} {'=' * 20}")

        tr_df = train_df.iloc[tr_idx].reset_index(drop=True)
        va_df = train_df.iloc[va_idx].reset_index(drop=True)

        tr_ds = FaceDataset(tr_df, transforms=get_train_transforms(CFG.img_size), is_test=False)
        va_ds = FaceDataset(va_df, transforms=get_valid_transforms(CFG.img_size), is_test=False)

        class_counts = tr_df["label"].value_counts().sort_index().values
        class_weights = 1.0 / class_counts
        sample_weights = tr_df["label"].map({i: w for i, w in enumerate(class_weights)}).values

        sampler = WeightedRandomSampler(
            weights=torch.DoubleTensor(sample_weights),
            num_samples=len(sample_weights),
            replacement=True,
        )

        train_loader = DataLoader(
            tr_ds,
            batch_size=CFG.batch_size,
            sampler=sampler,
            num_workers=CFG.num_workers,
            pin_memory=PIN_MEMORY,
            drop_last=True,
            persistent_workers=CFG.num_workers > 0,
        )
        valid_loader = DataLoader(
            va_ds,
            batch_size=CFG.batch_size,
            shuffle=False,
            num_workers=CFG.num_workers,
            pin_memory=PIN_MEMORY,
            drop_last=False,
            persistent_workers=CFG.num_workers > 0,
        )

        model = CustomFaceCNN().to(device)

        pos_count = max(1, int((tr_df["label"] == 1).sum()))
        neg_count = max(1, int((tr_df["label"] == 0).sum()))
        pos_weight = torch.tensor([neg_count / pos_count], dtype=torch.float32, device=device)

        criterion_train = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        criterion_valid = nn.BCEWithLogitsLoss()

        optimizer = AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=CFG.lr,
            epochs=CFG.epochs,
            steps_per_epoch=len(train_loader),
            pct_start=0.15,
            div_factor=10.0,
            final_div_factor=100.0,
        )

        scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
        ema = EMA(model, decay=CFG.ema_decay)

        best_score = -1.0
        best_threshold = 0.5
        best_state = None
        early_stop_counter = 0

        for epoch in range(1, CFG.epochs + 1):
            train_loss = train_one_epoch(
                model, train_loader, optimizer, scheduler, criterion_train, scaler, ema
            )

            ema.apply_shadow(model)
            val_loss, val_probs, val_targets = validate(model, valid_loader, criterion_valid)
            ema.restore(model)

            thr, val_f1 = find_best_threshold(val_targets, val_probs)

            print(
                f"Fold {fold} | Epoch {epoch:02d} | "
                f"train_loss={train_loss:.5f} | val_loss={val_loss:.5f} | "
                f"val_f1={val_f1:.5f} | thr={thr:.3f}"
            )

            if val_f1 > best_score:
                best_score = val_f1
                best_threshold = thr
                early_stop_counter = 0

                ema.apply_shadow(model)
                best_state = copy.deepcopy(model.state_dict())
                ema.restore(model)

                torch.save(
                    {
                        "model_state_dict": best_state,
                        "threshold": best_threshold,
                        "fold": fold,
                        "score": best_score,
                    },
                    os.path.join(CFG.out_dir, f"best_fold_{fold}.pth"),
                )
            else:
                early_stop_counter += 1

            if early_stop_counter >= CFG.patience:
                print(f"Early stopping on fold {fold}")
                break

        print(f"Best F1 on fold {fold}: {best_score:.5f} @ threshold={best_threshold:.3f}")

        model.load_state_dict(best_state)

        val_probs = predict_proba(model, valid_loader, tta=CFG.use_tta)
        oof_probs[va_idx] = val_probs

        fold_test_probs = predict_proba(model, test_loader, tta=CFG.use_tta)
        test_probs_folds.append(fold_test_probs)
        fold_thresholds.append(best_threshold)

        del model, optimizer, scheduler, scaler, ema, train_loader, valid_loader, tr_ds, va_ds
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    oof_target = train_df["label"].values.astype(np.int32)
    global_thr, global_f1 = find_best_threshold(oof_target, oof_probs)
    print(f"\nOOF F1 = {global_f1:.5f} @ threshold = {global_thr:.3f}")

    pd.DataFrame({
        "id": train_df["id"],
        "label": train_df["label"],
        "oof_prob": oof_probs,
    }).to_csv(os.path.join(CFG.out_dir, "oof_predictions.csv"), index=False)

    test_probs = np.mean(np.stack(test_probs_folds, axis=0), axis=0)
    test_pred = (test_probs >= global_thr).astype(np.int32)

    sub = pd.DataFrame({
        id_col: test_df["id"],
        target_col: test_pred,
    })
    sub.to_csv(os.path.join(CFG.out_dir, "submission.csv"), index=False)

    sub_proba = pd.DataFrame({
        id_col: test_df["id"],
        "probability": test_probs,
    })
    sub_proba.to_csv(os.path.join(CFG.out_dir, "submission_proba.csv"), index=False)

    print("\nГотово. Файлы сохранены в:", CFG.out_dir)
    print("Средний threshold по фолдам:", float(np.mean(fold_thresholds)))
    print("Глобальный threshold:", float(global_thr))


In [8]:
main()



Используемое устройство: cuda
AMP включен: True
Пути:
train_csv = /kaggle/input/datasets/tomatka/aifaceclassification/train_solution.csv
train_dir = /kaggle/input/datasets/tomatka/aifaceclassification/train_images
test_dir  = /kaggle/input/datasets/tomatka/aifaceclassification/test_images
out_dir   = /kaggle/working

Train size: 49999
Test  size: 10000
label
0    41499
1     8500
Name: count, dtype: int64

==================== FOLD 1/5 ====================


  0%|          | 0/1249 [00:00<?, ?it/s]

  0%|          | 0/313 [00:00<?, ?it/s]

Fold 1 | Epoch 01 | train_loss=1.27841 | val_loss=1.41433 | val_f1=0.29580 | thr=0.795


  0%|          | 0/1249 [00:00<?, ?it/s]

...

  0%|          | 0/313 [00:00<?, ?it/s]

Fold 5 | Epoch 30 | train_loss=0.11510 | val_loss=0.08309 | val_f1=0.93834 | thr=0.755
Best F1 on fold 5: 0.93836 @ threshold=0.820


  0%|          | 0/313 [00:00<?, ?it/s]

  0%|          | 0/313 [00:00<?, ?it/s]


OOF F1 = 0.94655 @ threshold = 0.660

Готово. Файлы сохранены в: /kaggle/working
Средний threshold по фолдам: 0.6529999999999998
Глобальный threshold: 0.6599999999999999


In [10]:
sub = pd.read_csv("/kaggle/working/submission.csv")
sub = sub.rename(columns={
    sub.columns[0]: "id",
    sub.columns[1]: "target_feature"
})
sub.to_csv("/kaggle/working/submission.csv", index=False)
print(sub.head())

     id  target_feature
0     0               1
1     1               1
2    10               0
3   100               0
4  1000               0
